# EgoBlur baseline ONLY -- for the RQ1 comparison

Runs **only EgoBlur** over a clip (no SAM 3, no depth). Use this on a clip you
**already processed with the method** -- do NOT re-run the method pipeline.
Writes `egoblur_eval.json` + `egoblur_sample.jpg`; paste both. The reviewer
merges this with the existing method log.

> T4/L4 GPU. EgoBlur weights are gated (.jit on Drive).

## Step 1 -- runtime

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print("CUDA:", torch.cuda.is_available())

## Step 2 -- install + clone EgoBlur

In [ ]:
!pip install -q opencv-python-headless matplotlib
import os, subprocess
if not os.path.isdir("/content/EgoBlur"):
    subprocess.run(["git","clone","-q","https://github.com/facebookresearch/EgoBlur.git",
                    "/content/EgoBlur"], check=True)
print("install + clone done")

## Step 3 -- config (edit me)

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import os
# The SAME clip you ran through the method:
VIDEO_PATH = "/content/drive/MyDrive/Licencjat/stock/stereo_right.mp4"
LENS       = "wide-angle"        # "wide-angle" | "fisheye" | "normal"
FACE_MODEL = "/content/drive/MyDrive/Licencjat/ego_blur_face_gen2.jit"
LP_MODEL   = "/content/drive/MyDrive/Licencjat/ego_blur_lp_gen2.jit"   # or None
ROTATE     = 0                   # 0/90/180/270 -- faces must be UPRIGHT for EgoBlur
FACE_THR, LP_THR = 0.674, 0.745
MIN_SIZE, MAX_SIZE = 1440, 2560  # detectron2 resize; larger -> better small-face recall
MAX_FRAMES = None                # None = whole clip
OUT_VIDEO  = "/content/egoblur_anon.mp4"
OUT_JSON   = "/content/egoblur_eval.json"
OUT_FIG    = "/content/egoblur_sample.jpg"
assert os.path.exists(VIDEO_PATH), "VIDEO_PATH not found -- mount Drive / fix path"
print("config OK | lens:", LENS, "| video:", os.path.basename(VIDEO_PATH))

## Step 4 -- helpers

In [ ]:
import sys, time, json, gc
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
def reset_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def peak_vram_gb():
    return round(torch.cuda.max_memory_allocated()/1e9,3) if torch.cuda.is_available() else None
print("helpers ready")

## Step 5 -- run EgoBlur + write egoblur_eval.json

In [ ]:
if "/content/EgoBlur" not in sys.path:
    sys.path.insert(0, "/content/EgoBlur")
from gen2.script.predictor import ClassID, EgoblurDetector, PATCH_INSTANCES_FIELDS
from gen2.script.detectron2.export.torchscript_patch import patch_instances

dev = "cuda" if torch.cuda.is_available() else "cpu"
resize = {"min_size_test": MIN_SIZE, "max_size_test": MAX_SIZE}
cfg = []
if FACE_MODEL and os.path.exists(FACE_MODEL):
    cfg.append(("face", EgoblurDetector(model_path=FACE_MODEL, device=dev,
        detection_class=ClassID.FACE, score_threshold=FACE_THR,
        nms_iou_threshold=0.3, resize_aug=resize)))
if LP_MODEL and os.path.exists(LP_MODEL):
    cfg.append(("licence plate", EgoblurDetector(model_path=LP_MODEL, device=dev,
        detection_class=ClassID.LICENSE_PLATE, score_threshold=LP_THR,
        nms_iou_threshold=0.3, resize_aug=resize)))
assert cfg, "no EgoBlur weights found at the given paths"

ROT = {0:None, 90:cv2.ROTATE_90_CLOCKWISE, 180:cv2.ROTATE_180, 270:cv2.ROTATE_90_COUNTERCLOCKWISE}
rflag = ROT[ROTATE]

def eb_detect(fr):
    t = torch.from_numpy(np.ascontiguousarray(fr.transpose(2,0,1))).to(dev)
    out = []
    for label, det in cfg:
        o = det.run(t)
        if o and isinstance(o[0], list) and (len(o[0])==0 or isinstance(o[0][0], list)):
            o = o[0]
        for b in o:
            if len(b) < 4:
                continue
            x1,y1,x2,y2 = (int(round(v)) for v in b[:4])
            out.append(((x1,y1,x2,y2), label))
    return out

def eb_redact(fr, ds):
    o = fr.copy(); h, w = fr.shape[:2]
    for (x1,y1,x2,y2), _ in ds:
        x1,y1 = max(0,x1), max(0,y1); x2,y2 = min(w,x2), min(h,y2)
        if x2<=x1 or y2<=y1:
            continue
        o[y1:y2, x1:x2] = (255,0,0)
    return o

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
ow, oh = (H, W) if ROTATE in (90,270) else (W, H)
writer = cv2.VideoWriter(OUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (ow, oh))
sample_idx = set(int(x) for x in np.linspace(0, max(0,(total or 120)-1), 3))

face_series = []; n = face_frames = lp_frames = nfc = nlc = 0
samples = {}; best = (-1, None, None)
reset_vram(); t0 = time.perf_counter()
with patch_instances(fields=PATCH_INSTANCES_FIELDS), torch.inference_mode():
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if rflag is not None:
            fr = cv2.rotate(fr, rflag)
        ds = eb_detect(fr); labels = [d[1] for d in ds]
        fc = labels.count("face"); lc = labels.count("licence plate")
        face_frames += (fc>0); lp_frames += (lc>0); nfc += fc; nlc += lc
        face_series.append(1 if fc>0 else 0)
        red = eb_redact(fr, ds); writer.write(red)
        if n in sample_idx:
            samples[n] = red.copy()
        if fc > best[0]:
            best = (fc, n, red.copy())
        n += 1
        if n % 200 == 0:
            print("  ...", n, "frames")
        if MAX_FRAMES and n >= MAX_FRAMES:
            break
cap.release(); writer.release(); dt = time.perf_counter() - t0

figs = []
if best[1] is not None:
    figs.append((best[1], best[2], "most faces"))
for k in sorted(samples):
    figs.append((k, samples[k], "frame " + str(k)))
if figs:
    fig, axs = plt.subplots(1, len(figs), figsize=(5*len(figs), 4))
    if len(figs) == 1:
        axs = [axs]
    for ax, (k, im, ttl) in zip(axs, figs):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title("EgoBlur -- " + ttl + " (f" + str(k) + ")"); ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT_FIG, dpi=110, bbox_inches="tight"); plt.show()

fs = face_series
if len(fs) > 600:
    step = (len(fs) + 599) // 600; fs = fs[::step]
EB = {"lens": LENS, "video_name": os.path.basename(VIDEO_PATH), "frames": n,
      "face_frames": face_frames, "lp_frames": lp_frames, "n_face": nfc, "n_lp": nlc,
      "face_rate": round(face_frames/n, 4) if n else 0.0,
      "fps": round(n/dt, 2) if dt else 0.0, "peak_vram_gb": peak_vram_gb(),
      "face_thr": FACE_THR, "lp_thr": LP_THR, "rotate": ROTATE,
      "min_size": MIN_SIZE, "max_size": MAX_SIZE,
      "max_faces_frame": best[1], "max_faces_count": best[0],
      "out": OUT_VIDEO, "face_series_sampled": fs}
with open(OUT_JSON, "w", encoding="utf-8") as fh:
    json.dump(EB, fh, indent=2, default=str)
print("=== PASTE THIS FILE:", OUT_JSON, "===")
print(json.dumps({k: v for k, v in EB.items() if k != "face_series_sampled"}, indent=2))
try:
    from google.colab import files; files.download(OUT_JSON)
except Exception as e:
    print("(download manually:", OUT_JSON, ")")